# Tutorial: Milestone 3 Render Image (Rust Backend)

Audience:
- Engineers validating real Rust `/render/image` behavior end-to-end.

Prerequisites:
- Run `uv sync` in this repository.
- Rust toolchain and `cargo` are available locally.

Learning goals:
- Start a real Rust daemon and exercise stateful/stateless render requests.
- Validate plane/pan/zoom/slab/LOD behavior and warning contracts.
- Verify file delivery safety and render-time patch ephemerality.


## Step 1 - Imports, repository resolution, and render fixture helper


In [7]:
from __future__ import annotations

import json
import os
import sys
import tempfile
from io import BytesIO
from pathlib import Path

from PIL import Image

from lucida.client import LucidaClient, LucidaClientError

cwd = Path.cwd().resolve()
if (cwd / 'MIGRATION.md').exists():
    REPO_ROOT = cwd
elif (cwd.parent.parent / 'MIGRATION.md').exists():
    REPO_ROOT = cwd.parent.parent
else:
    raise AssertionError('Could not locate repository root containing MIGRATION.md')

sys.path.insert(0, str(REPO_ROOT / 'tests'))
from rust_daemon import start_rust_daemon  # noqa: E402
from parity.data_setup import create_render_omezarr  # noqa: E402


def decode_size(payload_b64: str) -> tuple[int, int]:
    image = Image.open(BytesIO(__import__('base64').b64decode(payload_b64))).convert('RGBA')
    return image.size


def expect_client_error(fn, expected_fragment: str) -> None:
    try:
        fn()
        raise AssertionError(f'Expected error containing: {expected_fragment}')
    except Exception as exc:
        message = str(exc)
        assert expected_fragment in message, message


print(f'REPO_ROOT={REPO_ROOT}')


REPO_ROOT=/Users/austin/GitHub/lucida


## Step 2 - Build render dataset and start the Rust daemon


In [8]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-render-rust-'))
dataset_uri = create_render_omezarr(str(tmp_dir / 'render-demo.zarr'))

rust_daemon = start_rust_daemon(repo_root=REPO_ROOT, env=dict(os.environ))
client = LucidaClient(base_url=rust_daemon.base_url, backend='rust')

print('dataset_uri:', dataset_uri)
print('rust_base_url:', rust_daemon.base_url)


dataset_uri: /var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-render-rust-7gh8zw0x/render-demo.zarr
rust_base_url: http://127.0.0.1:55888


## Step 3 - Open dataset, create view, and validate stateful render contract


In [9]:
opened = client.open_dataset(uri=dataset_uri)
created = client.create_view(dataset_id=opened.dataset_summary.dataset_id, mode='2d')

stateful = client.render_image(
    view_id=created.view_state.view_id,
    width_px=96,
    height_px=64,
)

assert stateful.status == 'ok'
assert stateful.view_id == created.view_state.view_id
assert isinstance(stateful.state_version, int)
assert stateful.images[0].mime == 'image/png'
assert decode_size(stateful.images[0].bytes_base64 or '') == (96, 64)

print('view_id:', created.view_state.view_id)
print('stateful render_id:', stateful.render_id)
print('state_version:', stateful.state_version)


view_id: view_af74bc65df3346c9
stateful render_id: ren_0439804888a044b2
state_version: 0


## Step 4 - Validate stateless render and request guardrails


In [10]:
view_state = client.get_view(view_id=created.view_state.view_id).view_state

stateless = client.render_image(
    view_state=view_state,
    width_px=72,
    height_px=48,
)

assert stateless.status == 'ok'
assert stateless.view_id is None
assert stateless.state_version is None
assert decode_size(stateless.images[0].bytes_base64 or '') == (72, 48)

expect_client_error(
    lambda: client.render_image(
        view_id=created.view_state.view_id,
        view_state=view_state,
        width_px=64,
        height_px=48,
    ),
    'Exactly one of view_id or view_state must be provided',
)
expect_client_error(
    lambda: client.render_image(
        view_id=created.view_state.view_id,
        width_px=5000,
        height_px=48,
    ),
    'render_output_too_large',
)
expect_client_error(
    lambda: client.render_image(
        view_id=created.view_state.view_id,
        width_px=64,
        height_px=48,
        delivery='file_path',
        file_path='../bad.png',
    ),
    'render_output_path_invalid',
)

print('Stateless and guardrail assertions passed.')


Stateless and guardrail assertions passed.


## Step 5 - Exercise plane/pan/zoom/slab/LOD behavior and warning contracts


In [11]:
plane_updated = client.set_plane(view_id=created.view_state.view_id, plane='xz')
_ = client.pan(view_id=created.view_state.view_id, dx_px=20.0, dy_px=-8.0)
_ = client.zoom(view_id=created.view_state.view_id, factor=0.75)

client.update_view(
    view_id=created.view_state.view_id,
    patch=[
        {
            'op': 'replace',
            'path': '/selectors',
            'value': [
                {'axis': 'y', 'kind': 'range', 'start': 1, 'end_exclusive': 4, 'clamp': True},
                {'axis': 'c', 'kind': 'index', 'index': 0, 'clamp': True},
                {'axis': 't', 'kind': 'range', 'start': 0, 'end_exclusive': 1, 'clamp': True},
            ],
        },
        {
            'op': 'replace',
            'path': '/view_2d/slice/slab',
            'value': {'thickness_vox': 5, 'mode': 'single'},
        },
        {
            'op': 'replace',
            'path': '/performance',
            'value': {'lod_mode': 'fixed', 'fixed_level': 99},
        },
    ],
)

warning_render = client.render_image(
    view_id=created.view_state.view_id,
    width_px=64,
    height_px=48,
)
warning_codes = [warning.code for warning in warning_render.warnings]

assert 'slab_thickness_ignored' in warning_codes
assert 'lod_level_fallback_auto' in warning_codes
assert 'selector_reduced_to_index' in warning_codes

before = client.get_view(view_id=created.view_state.view_id).view_state
patched = client.render_image(
    view_id=created.view_state.view_id,
    width_px=64,
    height_px=48,
    overrides_json_patch=[
        {
            'op': 'replace',
            'path': '/selectors',
            'value': [{'axis': 'z', 'kind': 'index', 'index': 3, 'clamp': True}],
        }
    ],
)
after = client.get_view(view_id=created.view_state.view_id).view_state

assert patched.state_hash != before.state_hash
assert after.state_hash == before.state_hash
assert after.state_version == before.state_version

print('Warning bundle and ephemeral patch assertions passed.')


Warning bundle and ephemeral patch assertions passed.


## Step 6 - Validate file delivery under output root and cleanup


In [12]:
output_root = (REPO_ROOT / 'output').resolve()

explicit = client.render_image(
    view_id=created.view_state.view_id,
    width_px=48,
    height_px=36,
    delivery='file_path',
    file_path=f"snapshots/notebook-m3-{__import__('uuid').uuid4().hex}.png",
)
explicit_path = Path(explicit.images[0].file_path or '')
assert explicit_path.exists()
assert explicit_path.is_relative_to(output_root)
assert explicit.images[0].bytes_base64 is None

auto = client.render_image(
    view_id=created.view_state.view_id,
    width_px=40,
    height_px=30,
    delivery='file_path',
)
auto_path = Path(auto.images[0].file_path or '')
assert auto_path.exists()
assert auto_path.is_relative_to(output_root / 'snapshots')


In [13]:
if explicit_path.exists():
    explicit_path.unlink()
if auto_path.exists():
    auto_path.unlink()

client.close()
rust_daemon.stop()

print(json.dumps({
    'plane': plane_updated.view_state.view_2d.plane if plane_updated.view_state.view_2d else None,
    'warning_codes': warning_codes,
    'explicit_path': str(explicit_path),
    'auto_path': str(auto_path),
}, indent=2))

{
  "plane": "xz",
  "warning_codes": [
    "lod_level_fallback_auto",
    "slab_thickness_ignored",
    "selector_reduced_to_index"
  ],
  "explicit_path": "/Users/austin/GitHub/lucida/output/snapshots/notebook-m3-395034fbe5b74073a81650bb75404bef.png",
  "auto_path": "/Users/austin/GitHub/lucida/output/snapshots/ren_6bba8fe3af2e4aab.png"
}


## Expected output checks

After running all cells top-to-bottom, verify:
- Rust daemon starts and base URL is printed.
- Stateful render returns valid PNG dimensions and non-empty `state_hash`.
- Stateless render omits `view_id` and `state_version`.
- Guardrail checks raise expected errors: `invalid_render_request`, `render_output_too_large`, `render_output_path_invalid`.
- Warning render contains `slab_thickness_ignored`, `lod_level_fallback_auto`, and `selector_reduced_to_index`.
- Render-time patch changes response hash but does not persist ViewState changes.
- File-delivery renders are written under `output/snapshots/` and cleaned up.
